# 01 — Preparação dos dados (Pesquisa OD 2017)

**Global Solution 2026.2 · MLAM · Citizen Tech** · **Integrante:** Davi Zuolo — RM: ______

**Problema:** em São Paulo, a oferta de ônibus nem sempre está onde a demanda está. Para planejar melhor é preciso saber **quem depende do ônibus para trabalhar** e **quanto tempo essa viagem custa**.

**Base:** Pesquisa Origem e Destino 2017, da Companhia do Metropolitano de São Paulo (Metrô-SP). É uma pesquisa domiciliar real, com 183.092 registros e 128 variáveis, em que cada linha é uma viagem de uma pessoa num dia útil.
- Fonte original: Metrô-SP, portal de transparência (`transparencia.metrosp.com.br`, Pesquisa Origem e Destino 2017).
- Endereço de acesso usado (espelho público do pacote R `odbr`, CRAN): `https://github.com/hsvab/odbr/releases/download/v0.1.0/od_sao_paulo_2017_not_harmonized.csv.gz`
- Acesso em 09/11/2026. Dicionário de variáveis: `data/dicionario/od2017_dicionario_pt.csv` na raiz do repositório.

Este notebook transforma o arquivo bruto na base de análise `dados/od2017_sp_trabalho.csv`. Todas as etapas estão em código e são reproduzíveis.

In [1]:
import pathlib, urllib.request
import numpy as np
import pandas as pd

URL = "https://github.com/hsvab/odbr/releases/download/v0.1.0/od_sao_paulo_2017_not_harmonized.csv.gz"
BRUTO = pathlib.Path("../../data/raw/od_sao_paulo_2017_not_harmonized.csv.gz")
if not BRUTO.exists():                      # baixa da fonte se o arquivo não estiver no repositório
    BRUTO = pathlib.Path("dados/od_sao_paulo_2017_bruto.csv.gz")
    BRUTO.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL, BRUTO)

od = pd.read_csv(BRUTO, sep=";")
print("registros:", len(od), "| colunas:", od.shape[1])

registros: 183092 | colunas: 128


## Unidade de análise
Uma linha por **trabalhador**: a primeira viagem do dia com destino "trabalho", com origem e destino dentro do município de São Paulo.

Por que uma viagem por pessoa: a ida e a volta da mesma pessoa são quase iguais (mesma distância, mesmo modo). Manter as duas faria a mesma informação contar duas vezes e violaria a independência das observações, que os testes e as regressões pressupõem.

## Etapas de limpeza (funil)
| Etapa | Regra | Motivo |
|---|---|---|
| 1 | registro é uma viagem (`modo1` preenchido) | a base também lista pessoas que não viajaram |
| 2 | `muni_o == 36` e `muni_d == 36` | recorte: município de São Paulo |
| 3 | `motivo_d` em {1, 2, 3} | destino é trabalho (indústria, comércio, serviços) |
| 4 | `cd_ativi == 1` | pessoa com trabalho regular |
| 5 | sem nulos nas variáveis usadas e `renda_fa > 0` | valores inválidos; renda zero impede o logaritmo |
| 6 | primeira viagem a trabalho de cada pessoa | independência das observações |

In [2]:
USADAS = ["renda_fa", "qt_auto", "idade", "distancia", "duracao", "vinc1", "ocup1", "no_morad"]
etapas = [
    ("1. registro é uma viagem", lambda d: d.modo1.notna()),
    ("2. origem e destino em São Paulo", lambda d: (d.muni_o == 36) & (d.muni_d == 36)),
    ("3. destino é trabalho", lambda d: d.motivo_d.isin([1, 2, 3])),
    ("4. trabalho regular", lambda d: d.cd_ativi == 1),
    ("5. sem nulos e renda > 0", lambda d: d[USADAS].notna().all(axis=1) & (d.renda_fa > 0)),
    ("6. primeira viagem a trabalho da pessoa", lambda d: ~d.sort_values("n_viag").duplicated("id_pess").reindex(d.index)),
]
base, funil = od, [("0. base bruta", len(od))]
for nome, regra in etapas:
    base = base[regra(base)]
    funil.append((nome, len(base)))
funil = pd.DataFrame(funil, columns=["etapa", "linhas"])
funil["removidas"] = -funil.linhas.diff().fillna(0).astype(int)
display(funil)

,etapa,linhas,removidas
0,0. base bruta,183092,0
1,1. registro é uma viagem,157992,25100
2,2. origem e destino em São Paulo,106145,51847
3,3. destino é trabalho,26778,79367
4,4. trabalho regular,25228,1550
5,5. sem nulos e renda > 0,24576,652
6,6. primeira viagem a trabalho da pessoa,19462,5114


## Variáveis derivadas
| Variável | Definição | Tipo |
|---|---|---|
| `usa_onibus` | 1 se o modo principal é ônibus do município de SP (`modoprin == 4`) | binária (alvo da logística) |
| `duracao` | duração da viagem em minutos (original) | quantitativa |
| `distancia_km` | distância em linha reta entre origem e destino, em km | quantitativa |
| `renda_pc` | renda familiar mensal ÷ moradores do domicílio | quantitativa |
| `log_renda_pc` | logaritmo natural de `renda_pc` | quantitativa |
| `qt_auto` | automóveis no domicílio (original) | quantitativa |
| `idade` | anos (original) | quantitativa |
| `clt` | 1 se assalariado com carteira (`vinc1 == 1`) | binária |
| `pico` | 1 se saiu entre 6h–8h ou 17h–19h | binária |
| `ocupacao` | grupo de ocupação (`ocup1`), com grupos de menos de 100 pessoas unidos em 11 = "outras" | categórica |
| `renda_declarada` | 1 se a renda foi declarada (`cd_renfa == 1`); 0 se foi atribuída pela pesquisa | binária (controle de qualidade) |
| `n_trechos` | quantidade de modos usados na viagem (1 a 4) | quantitativa |
| `modo` | modo principal agrupado: ônibus, carro, metrô/trem, a pé, outros | categórica |

In [3]:
raras = base.ocup1.map(base.ocup1.value_counts()) < 100
grupo_modo = {4: "ônibus", 9: "carro", 10: "carro", 1: "metrô/trem", 2: "metrô/trem", 16: "a pé"}
tratada = pd.DataFrame({
    "id_fam": base.id_fam.astype("int64"),
    "usa_onibus": (base.modoprin == 4).astype(int),
    "modo": base.modoprin.map(grupo_modo).fillna("outros"),
    "duracao": base.duracao,
    "distancia_km": base.distancia / 1000,
    "renda_pc": base.renda_fa / base.no_morad,
    "log_renda_pc": np.log(base.renda_fa / base.no_morad),
    "qt_auto": base.qt_auto.astype(int),
    "idade": base.idade.astype(int),
    "clt": (base.vinc1 == 1).astype(int),
    "pico": base.h_saida.isin([6, 7, 8, 17, 18, 19]).astype(int),
    "ocupacao": base.ocup1.mask(raras, 11).astype(int),
    "renda_declarada": (base.cd_renfa == 1).astype(int),
    "n_trechos": base[["modo1", "modo2", "modo3", "modo4"]].notna().sum(axis=1),
}).round(4).reset_index(drop=True)

assert tratada.notna().all().all() and (tratada.duracao > 0).all() and (tratada.distancia_km > 0).all()
tratada.to_csv("dados/od2017_sp_trabalho.csv", index=False)
print("salvo: dados/od2017_sp_trabalho.csv |", tratada.shape)
display(tratada.head())

salvo: dados/od2017_sp_trabalho.csv | (19462, 14)


,id_fam,usa_onibus,modo,duracao,distancia_km,renda_pc,log_renda_pc,qt_auto,idade,clt,pico,ocupacao,renda_declarada,n_trechos
0,100011,0,a pé,10.0,0.6391,1366.288,7.2199,0,59,1,0,4,0,1
1,100011,0,metrô/trem,50.0,6.6518,1366.288,7.2199,0,21,0,0,4,0,1
2,100021,0,a pé,10.0,0.3683,800.000,6.6846,0,37,1,1,6,1,1
3,100021,0,a pé,10.0,0.2780,800.000,6.6846,0,19,1,1,6,1,1
4,100021,0,metrô/trem,20.0,1.8426,800.000,6.6846,0,18,1,0,5,1,1


In [4]:
display(tratada.describe().T.round(2))
display(tratada.modo.value_counts(normalize=True).rename("proporção").round(3).to_frame())
print(f"renda atribuída pela pesquisa (não declarada): {1 - tratada.renda_declarada.mean():.1%} dos trabalhadores")

,count,mean,std,min,25%,50%,75%,max
id_fam,19462.0,16907581.67,10935778.38,100011.00,7113101.00,15708551.00,26908176.00,51714091.00
usa_onibus,19462.0,0.22,0.41,0.00,0.00,0.00,0.00,1.00
duracao,19462.0,39.47,31.53,1.00,15.00,30.00,60.00,210.00
distancia_km,19462.0,6.35,6.34,0.00,1.47,4.48,9.17,43.95
renda_pc,19462.0,2476.86,2501.86,45.00,966.17,1675.00,3071.29,52000.00
log_renda_pc,19462.0,7.47,0.82,3.81,6.87,7.42,8.03,10.86
qt_auto,19462.0,0.94,0.83,0.00,0.00,1.00,1.00,8.00
idade,19462.0,41.07,13.67,14.00,30.00,40.00,51.00,98.00
clt,19462.0,0.57,0.49,0.00,0.00,1.00,1.00,1.00
pico,19462.0,0.61,0.49,0.00,0.00,1.00,1.00,1.00


,proporção
modo,
carro,0.329
ônibus,0.220
a pé,0.191
metrô/trem,0.188
outros,0.071


renda atribuída pela pesquisa (não declarada): 63.2% dos trabalhadores


## Observações sobre a qualidade dos dados
- **Outliers.** O corte por IQR não é aplicado aqui: ele é feito no notebook de regressões lineares, sobre `duracao` e `distancia_km`, com contagem das linhas removidas e comparação do resultado com e sem o corte. Na regressão logística os valores extremos de renda e carros são mantidos, porque não são erros: são justamente os casos que distinguem quem usa ônibus.
- **Renda atribuída.** Em boa parte das famílias a renda não foi declarada e a pesquisa a estimou (critério Brasil ou média da zona). A coluna `renda_declarada` permite testar se isso altera as conclusões.
- **Duração declarada.** As pessoas informam a duração em múltiplos de 5 minutos, o que gera "degraus" nos gráficos.
- **Ano.** A pesquisa é de 2017, anterior à pandemia. Serve como linha de base, não como retrato de hoje.
- **Identificação.** A base não tem nome, endereço nem documento. `id_fam` é o código anônimo da família na pesquisa, mantido só para que moradores da mesma casa não sejam separados entre treino e teste.